<a href="https://colab.research.google.com/github/neelay8975/GenAi_prac1/blob/main/Prac_5_genAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Transfer Learning for Text Classification

This notebook demonstrates how to apply transfer learning using a pre-trained language model for text classification. The goal is to categorize news articles into 'sports', 'politics', and 'technology' with limited labeled data.

### What is Transfer Learning?

Transfer learning is a machine learning technique where a model trained on one task is re-purposed for a second related task. In Natural Language Processing (NLP), this often involves using a large pre-trained language model (like BERT, RoBERTa, DistilBERT) that has learned general language understanding from a massive corpus of text. This pre-trained model is then fine-tuned on a smaller, specific dataset for a downstream task, such as text classification.

### 1. Install Necessary Libraries

We'll need the `transformers` library from Hugging Face for pre-trained models and `datasets` for efficient data handling.

In [2]:
# Install Hugging Face Transformers and Datasets libraries
%pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

### 2. Load and Prepare Data

Since no specific dataset was provided, we'll create a dummy dataset for demonstration. In a real-world scenario, you would load your news articles and their corresponding labels here. The data should be in a format suitable for text classification, typically with 'text' and 'label' columns.

In [4]:
# Dummy dataset (replace with your actual data loading)
data = {
    'text': [
        'Football team wins championship in a thrilling match.',
        'New bill proposed to address economic inequality.',
        'AI breakthroughs in neural networks reported.',
        'Election results show a surprising turn of events.',
        'Basketball playoffs heating up in the final quarter.',
        'New quantum computing advancements announced.',
        'Government approves new infrastructure project.',
        'Tennis star clinches grand slam title.',
        'SpaceX launches new satellite into orbit.',
        'Political debate focuses on climate change.',
        'Local team suffers a heavy defeat.',
        'Technology firm unveils new smartphone.',
        'Legislators vote on controversial healthcare reform.',
        'Scientists discover new exoplanet.',
        'World Cup qualifiers begin next month.'
    ],
    'label': [
        'sports',
        'politics',
        'technology',
        'politics',
        'sports',
        'technology',
        'politics',
        'sports',
        'technology',
        'politics',
        'sports',
        'technology',
        'politics',
        'technology',
        'sports'
    ]
}

df = pd.DataFrame(data)

# Map labels to integers
label_to_id = {'sports': 0, 'politics': 1, 'technology': 2}
id_to_label = {0: 'sports', 1: 'politics', 2: 'technology'}
df['label_id'] = df['label'].map(label_to_id)

# Convert to Hugging Face Dataset format
hf_dataset = Dataset.from_pandas(df)

# Split into training and test sets (e.g., 80% train, 20% test)
train_test_split = hf_dataset.train_test_split(test_size=0.2, seed=42)
dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

print(f"Training examples: {len(dataset_dict['train'])}")
print(f"Test examples: {len(dataset_dict['test'])}")
print(dataset_dict)


Training examples: 12
Test examples: 3
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 12
    })
    test: Dataset({
        features: ['text', 'label', 'label_id'],
        num_rows: 3
    })
})


### 3. Load Pre-trained Tokenizer and Model

We'll use `distilbert-base-uncased` for this example, which is a smaller, faster version of BERT suitable for fine-tuning. The tokenizer converts text into numerical input IDs that the model can understand.

In [5]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding=True)

tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Remove the original text and label columns, keep only the tokenized inputs and label_id
tokenized_datasets = tokenized_datasets.remove_columns(['text', 'label'])
tokenized_datasets = tokenized_datasets.rename_column('label_id', 'labels')
tokenized_datasets.set_format('torch')

print(tokenized_datasets)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3
    })
})


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(label_to_id))

# Define the metric for evaluation
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 4. Fine-tune the Model

Now we'll fine-tune the pre-trained model on our specific text classification task using the `Trainer` API.

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # Corrected argument name
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none" # Disable logging to external services like W&B
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    # tokenizer=tokenizer, # Removed this argument
    compute_metrics=compute_metrics,
)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.141520,0.000000
2,No log,1.144209,0.000000
3,No log,1.145161,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6, training_loss=1.0378390153249104, metrics={'train_runtime': 46.8832, 'train_samples_per_second': 0.768, 'train_steps_per_second': 0.128, 'total_flos': 102457080792.0, 'train_loss': 1.0378390153249104, 'epoch': 3.0})

### 5. Evaluate the Fine-tuned Model

After training, we can evaluate the model's performance on the test set.

In [12]:
eval_results = trainer.evaluate()
print(eval_results)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,1.141520,3,0.000000


{'eval_loss': 1.1415201425552368, 'eval_accuracy': 0.0}
